# AgentMorph — Stage 2: seed Gemini paraphrase cache

One-shot setup for Stage 2 rules 2, 3, 5. Runs on a **CPU runtime** (no GPU needed — this is pure API calls + cache file writes).

**What it does:**

1. Clones/pulls the repo.
2. Installs the `[gemini]` extra so `google-generativeai` is available.
3. Runs `scripts/generate_paraphrases.py` against your `GEMINI_API_KEY`.
4. Commits the 3 resulting `runs/paraphrase_cache/*.jsonl` files to `main`.

**Total Gemini calls:** 54 (30 + 20 + 4). Well under the free-tier 1,500/day limit.

**Prereq:** a free Gemini API key from https://aistudio.google.com/app/apikey.

## 1. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 2. Install the `[gemini]` extra

Just `google-generativeai` — no torch, no transformers needed for this step.

In [ ]:
!pip install -q -e /content/AgentMorph[gemini]

## 3. Gemini API key

Recommended: use Colab Secrets (🔑 icon in left sidebar) to store `GEMINI_API_KEY` — then nothing leaks into the notebook cells. Alternatively paste it inline and **clear the cell** before sharing the notebook.

Get a free key at https://aistudio.google.com/app/apikey (takes < 2 min).

In [ ]:
import os

# Option A (recommended) — Colab Secrets
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('GEMINI_API_KEY loaded from Colab Secrets.')
except Exception as exc:
    print('Colab Secrets not available or GEMINI_API_KEY not set there.')
    print('Fall back: paste your key below and re-run the cell.')

# Option B (fallback) — uncomment and paste inline
# os.environ['GEMINI_API_KEY'] = 'PASTE_YOUR_KEY_HERE'

assert os.environ.get('GEMINI_API_KEY'), 'GEMINI_API_KEY not set.'

## 4. (Optional) Dry-run to verify what will be seeded

No API calls — just lists the 54 entries that would be generated.

In [ ]:
!cd /content/AgentMorph && python scripts/generate_paraphrases.py --dry-run

## 5. Actual cache seeding

~2 min, 54 live Gemini API calls. Writes JSONL files under `/content/AgentMorph/runs/paraphrase_cache/`.

In [ ]:
!cd /content/AgentMorph && python scripts/generate_paraphrases.py

## 6. Verify the cache is populated

Expect 3 JSONL files: `synonym_robustness.jsonl` (20 lines), `schema_paraphrase_invariance.jsonl` (30 lines), `refusal_consistency.jsonl` (4 lines).

In [ ]:
import pathlib, json
cache_dir = pathlib.Path('/content/AgentMorph/runs/paraphrase_cache')
for p in sorted(cache_dir.glob('*.jsonl')):
    n = sum(1 for _ in p.open())
    print(f'{p.name}: {n} entries')

## 7. Commit the cache to the repo

The Stage 3 sweep reads the cache from a known commit on `main` and runs `offline=True` — so the cache must be committed for reproducibility.

**You'll need to configure git credentials once** (only in a fresh Colab runtime). Easiest path: Colab's built-in git integration — `from google.colab import auth; auth.authenticate_user()` — OR use a personal access token pasted as the password.

In [ ]:
# Configure git user if not already set (Colab's default isn't suitable for push).
!cd /content/AgentMorph && git config user.email 'you@example.com'
!cd /content/AgentMorph && git config user.name 'AgentMorph Seed Bot'

# Stage + commit + push.
!cd /content/AgentMorph && git add runs/paraphrase_cache/
!cd /content/AgentMorph && git status --short
!cd /content/AgentMorph && git commit -m "paraphrase: seed cache for Stage 2 rules 2, 3, 5"
!cd /content/AgentMorph && git push origin main

## Done.

After the push, the paraphrase cache is on `main` and any future Stage 3 sweep (`python -m agentmorph.runner --stage3 ...`) will read entries from it in `offline=True` mode — zero live Gemini calls during the long T4 sweep.

**Next step:** the Stage 3 runner extension (`--stage3` flag) gets shipped in a separate commit. Once that lands, run `notebooks/stage3_dress_rehearsal.ipynb` for the 200-pair smoke, then `notebooks/stage3_full_sweep.ipynb` for the Apr 28 overnight 1,000-pair run.